# Data Preparation

In [42]:
# Imports

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, vstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer



In [43]:
# Load Data

df = pd.read_csv('../01_data/01_raw_data/customer_original.csv')

df


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [44]:
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


In [45]:
df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

In [46]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

# Data Preparation on Order-Level Dataset

In [47]:
# Rename

df = df.rename(columns 
               = {
                   'Customer ID': 'CustomerID',
                   'Price': 'UnitPrice',
                   'Invoice': 'InvoiceNo'
}
)

In [48]:
# New Formats

df['CustomerID'] = df['CustomerID'].astype(str).fillna('Unknown')

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)


### New Features

In [49]:
# New Variables (Date, Time, Revenue etc.)

# Date & Time
df['invoice_date'] = df['InvoiceDate'].dt.date
df['invoice_time'] = df['InvoiceDate'].dt.time
df['invoice_hour'] = df['InvoiceDate'].dt.hour

df['invoice_year'] = df['InvoiceDate'].dt.year
df['invoice_month'] = df['InvoiceDate'].dt.month
df['invoice_weekday_num'] = df['InvoiceDate'].dt.dayofweek  # Monday=0, Sunday=6
df['invoice_weekday'] = df['InvoiceDate'].dt.day_name()
df['is_weekend'] = df['invoice_weekday_num'].isin([5, 6])
df['invoice_calendarweek'] = df['InvoiceDate'].dt.isocalendar().week   


# Time of Day with 4 Categories
def get_time_of_day(hour):
    if 0 <= hour < 6:
        return "night"
    elif 6 <= hour < 10:
        return "morning"
    elif 10 <= hour < 14:
        return "midday"
    elif 14 <= hour < 18:
        return "afternoon"
    else:
        return "evening"

df['invoice_time_5cat'] = df['invoice_hour'].apply(get_time_of_day)


# Revenue 
df['RevenueLine'] = df['Quantity']*df['UnitPrice']

df[
    [
        'Quantity', 
        'UnitPrice',
        'RevenueLine',
    ]
].head()

df[
    [
        'InvoiceDate',
        'invoice_date',
        'invoice_calendarweek',
        'invoice_weekday',
        'is_weekend',
        'invoice_time',
        'invoice_hour',
        'invoice_time_5cat',
        'Quantity',
        'UnitPrice',
        'RevenueLine',
    ]
].head()

,InvoiceDate,invoice_date,invoice_calendarweek,invoice_weekday,is_weekend,invoice_time,invoice_hour,invoice_time_5cat,Quantity,UnitPrice,RevenueLine
0,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,12,6.95,83.4
1,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,12,6.75,81.0
2,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,12,6.75,81.0
3,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,48,2.10,100.8
4,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,24,1.25,30.0


In [50]:
# Descriptions

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    max_features=100
)

descriptions = df["Description"].fillna("").astype(str)

X_text_line = vectorizer.fit_transform(descriptions)

customer_ids = df['CustomerID'].to_numpy()
unique_customer_ids = np.sort(np.unique(customer_ids))

customer_text_vectors = []

for customer_id in unique_customer_ids:
    customer_mask = customer_ids == customer_id
    customer_vector = X_text_line[customer_mask].mean(axis=0)

    customer_text_vectors.append(csr_matrix(customer_vector))

X_text_customer = vstack(customer_text_vectors)

In [51]:
feature_names = vectorizer.get_feature_names_out()

print(feature_names)
print(f"Number Text Features: {len(feature_names)}")

['12' '20' '60' 'antique' 'assorted' 'bag' 'bird' 'birthday' 'black'
 'blue' 'bottle' 'bowl' 'box' 'bunting' 'cake' 'cake cases' 'candle'
 'candles' 'card' 'cases' 'ceramic' 'charlotte' 'charlotte bag'
 'christmas' 'colour' 'cream' 'cutlery' 'decoration' 'design' 'dolly'
 'door' 'doormat' 'fairy' 'fairy cake' 'feltcraft' 'flower' 'frame'
 'garden' 'girl' 'glass' 'green' 'hanging' 'hanging heart' 'heart'
 'holder' 'home' 'hot' 'hot water' 'ivory' 'jumbo' 'jumbo bag' 'kit'
 'large' 'light' 'light holder' 'lights' 'love' 'lunch' 'lunch bag'
 'metal' 'metal sign' 'mini' 'mug' 'pack' 'paisley' 'paper' 'party' 'pink'
 'polkadot' 'red' 'red retrospot' 'red spotty' 'regency' 'retro'
 'retrospot' 'rose' 'set' 'sign' 'silver' 'skull' 'small' 'spaceboy'
 'spot' 'spotty' 'star' 'strawberry' 'tea' 'tin' 'trinket' 'union'
 'vintage' 'water' 'water bottle' 'white' 'wicker' 'wood' 'wooden'
 'woodland' 'wrap' 'zinc']
Number Text Features: 100


### Transactions Filtering - Purchases, Returns, Cancellations

In [52]:
# Helper variables
df["is_cancellation_invoice"] = df["InvoiceNo"].str.startswith(
    "C",
    na=False
)
df["is_return_quantity"] = df["Quantity"].lt(0)
df["is_non_positive_price"] = df["UnitPrice"].le(0)


# df Purchases
purchases = df[
    df["CustomerID"].notna()
    & df["InvoiceDate"].notna()
    & df["Description"].notna()
    & df["Quantity"].gt(0)
    & df["UnitPrice"].gt(0)
    & ~df["is_cancellation_invoice"]
].copy()

print(purchases.shape)
purchases.head()

(1041670, 22)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,invoice_date,invoice_time,...,invoice_month,invoice_weekday_num,invoice_weekday,is_weekend,invoice_calendarweek,invoice_time_5cat,RevenueLine,is_cancellation_invoice,is_return_quantity,is_non_positive_price
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,83.4,False,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,81.0,False,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,81.0,False,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,100.8,False,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,30.0,False,False,False


In [53]:
purchases = df[
    df['CustomerID'].notna()
    & df['InvoiceDate'].notna()
    & df['Description'].notna()
    & df['Quantity'].gt(0)
    & df['UnitPrice'].gt(0)
    & ~df['InvoiceNo'].astype(str).str.startswith("C", na=False)
].copy()

# purchases.describe()

In [54]:
# df Returns

returns = df[
    df['CustomerID'].notna()
    & (
        df['is_cancellation_invoice']
        | df['is_return_quantity']
    )
].copy()

returns['return_value'] = (
    returns['Quantity'] * returns['UnitPrice']
)

returns.shape

(22951, 23)

In [55]:
pd.crosstab(
    df['is_cancellation_invoice'],
    df['is_return_quantity'],
    margins=True
)

is_return_quantity,False,True,All
is_cancellation_invoice,,,
False,1044420,3457,1047877
True,1,19493,19494
All,1044421,22950,1067371


In [56]:
# df Cancellations

cancellations = df[df['UnitPrice'] < 0].copy()

cancellations[
    [
        'InvoiceNo',
        'StockCode',
        'Description',
        'Quantity',
        'UnitPrice',
        'CustomerID',
    ]
].head(20)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
179403,A506401,B,Adjust bad debt,1,-53594.36,Unknown
276274,A516228,B,Adjust bad debt,1,-44031.79,Unknown
403472,A528059,B,Adjust bad debt,1,-38925.87,Unknown
825444,A563186,B,Adjust bad debt,1,-11062.06,Unknown
825445,A563187,B,Adjust bad debt,1,-11062.06,Unknown


In [57]:
print("Number Customers:", purchases.groupby('CustomerID').count().shape)
print("Shape :", X_text_customer.shape)


Number Customers: (5879, 21)
Shape : (5943, 100)


In [58]:
text_feature_map = pd.DataFrame({
    "column_index": range(len(feature_names)),
    "term": feature_names
})

#text_feature_map.head(50)

In [59]:
# Overview Datasets

print('##################################################################################################')
print('Purchases Dataset:', purchases.shape)
print('Purchases Dataset:', purchases.columns)
print('##################################################################################################')
print('Returns Dataset:', returns.shape)
print('Returns Dataset:', returns.columns)
print('##################################################################################################')
print('Cancellations Dataset:', cancellations.shape)
print('Cancellations Dataset:', cancellations.columns)

##################################################################################################
Purchases Dataset: (1041670, 22)
Purchases Dataset: Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'invoice_date', 'invoice_time',
       'invoice_hour', 'invoice_year', 'invoice_month', 'invoice_weekday_num',
       'invoice_weekday', 'is_weekend', 'invoice_calendarweek',
       'invoice_time_5cat', 'RevenueLine', 'is_cancellation_invoice',
       'is_return_quantity', 'is_non_positive_price'],
      dtype='str')
##################################################################################################
Returns Dataset: (22951, 23)
Returns Dataset: Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'invoice_date', 'invoice_time',
       'invoice_hour', 'invoice_year', 'invoice_month', 'invoice_weekday_num',
       'invoice_weekday', 'is_week

### Save Datasets on Order-Level Information

In [60]:
## Purchases (main dataset)
purchases.to_csv('../01_data/02_processed_data/purchases_line.csv', index=False)

## Returns
returns.to_csv('../01_data/02_processed_data/returns_line.csv', index=False)

## Cancellations
cancellations.to_csv('../01_data/02_processed_data/cancellations_line.csv', index=False)

## Text data
text_feature_map.to_csv(
    "../03_results/tfidf_feature_vocabulary.csv",
    index=False
)


# Data Preparation on Customer-Level Dataset

In [61]:
## Purchases (main dataset)

analysis_date = purchases["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm_customer = (
    purchases
    .groupby("CustomerID")
    .agg(
        recency=(
            "InvoiceDate",
            lambda dates: (analysis_date - dates.max()).days
        ),
        frequency=(
            "InvoiceNo",
            "nunique"
        ),
        monetary=(
            "RevenueLine",
            "sum"
        ),
    )
    .reset_index()
)

print(rfm_customer.shape)
rfm_customer.head()


(5879, 4)


,CustomerID,recency,frequency,monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,5633.32
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


In [62]:
customer_text = (
    purchases
    .groupby("CustomerID")["Description"]
    .agg(" ".join)
    .reset_index(name="customer_product_text")
)

print(customer_text.shape)
customer_text.head()

(5879, 2)


,CustomerID,customer_product_text
0,12346.0,This is a test product. This is a test product...
1,12347.0,PINK REGENCY TEACUP AND SAUCER ROSES REGENCY T...
2,12348.0,PACK OF 72 SKULL CAKE CASES 60 TEATIME FAIRY C...
3,12349.0,PLASTERS IN TIN WOODLAND ANIMALS PLASTERS IN T...
4,12350.0,CHOCOLATE THIS WAY METAL SIGN METAL SIGN NEIGH...


### Text Preparation

In [63]:
# TfidfVectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    max_features=100
)

X_text_customer = vectorizer.fit_transform(
    customer_text["customer_product_text"]
)

customer_ids_text = customer_text["CustomerID"].to_numpy()

components_to_test = [5, 10, 15, 20, 30]

# TruncatedSVD
for n_components in components_to_test:
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_reduced = svd.fit_transform(X_text_customer)

    print(
        f"{n_components:>2} components | "
        f"explained variance: {svd.explained_variance_ratio_.sum():.3f}"
    )

 5 components | explained variance: 0.292
10 components | explained variance: 0.439
15 components | explained variance: 0.538
20 components | explained variance: 0.615
30 components | explained variance: 0.725


In [64]:
# Decision for a Test with 15 Components

svd = TruncatedSVD(n_components=15, n_iter=7, random_state=42)
X_text_svd = svd.fit_transform(X_text_customer)

text_svd_df = pd.DataFrame(
    X_text_svd,
    columns=[f"text_svd_{i:02d}" for i in range(X_text_svd.shape[1])]
)

def show_svd_terms(svd_model, feature_names, top_n=10):
    rows = []

    for i, component in enumerate(svd_model.components_):
        top_positive_idx = component.argsort()[-top_n:][::-1]
        top_negative_idx = component.argsort()[:top_n]

        rows.append({
            "component": f"text_svd_{i}",
            "explained_variance": svd_model.explained_variance_ratio_[i],
            "top_positive_terms": ", ".join(feature_names[top_positive_idx]),
            "top_negative_terms": ", ".join(feature_names[top_negative_idx])
        })

    return pd.DataFrame(rows)

feature_names = np.array(vectorizer.get_feature_names_out())

svd_terms = show_svd_terms(
    svd_model=svd,
    feature_names=feature_names,
    top_n=20
)
display(svd_terms)

,component,explained_variance,top_positive_terms,top_negative_terms
0,text_svd_0,0.045301,"set, heart, red, bag, christmas, vintage, retr...","charlotte bag, charlotte, skull, 20, girl, fai..."
1,text_svd_1,0.091661,"bag, lunch, jumbo, lunch bag, jumbo bag, set, ...","heart, hanging, holder, light, light holder, m..."
2,text_svd_2,0.055876,"bag, jumbo, jumbo bag, lunch bag, sign, lunch,...","set, christmas, cake, cases, pack, cake cases,..."
3,text_svd_3,0.051528,"sign, metal sign, metal, hot, hot water, water...","light, light holder, holder, bag, heart, hangi..."
4,text_svd_4,0.048034,"water bottle, hot water, water, bottle, hot, c...","sign, metal sign, metal, cake, cases, cake cas..."
5,text_svd_5,0.038506,"christmas, vintage, sign, metal, wooden, card,...","cake, red, spotty, ceramic, red spotty, pink, ..."
6,text_svd_6,0.030031,"cake, cake cases, cases, christmas, heart, 60,...","set, glass, box, red, card, blue, candle, vint..."
7,text_svd_7,0.028470,"red, retrospot, red retrospot, christmas, spot...","lunch, lunch bag, spaceboy, design, light, set..."
8,text_svd_8,0.025750,"jumbo, jumbo bag, glass, regency, vintage, sil...","lunch, heart, lunch bag, feltcraft, spaceboy, ..."
9,text_svd_9,0.023973,"pink, feltcraft, white, heart, blue, garden, d...","light, glass, light holder, holder, retrospot,..."


The SVD components capture latent product-preference patterns such as bags, home décor, seasonal and gift-related items, baking accessories, and selected product designs. The components are retained as technical features and are not treated as mutually exclusive product categories.

In [65]:
text_svd_df = pd.DataFrame(
    X_text_svd,
    columns=[f"text_svd_{i:02d}" for i in range(X_text_svd.shape[1])]
)

text_svd_df.insert(0, "CustomerID", customer_ids_text)

### RFM

In [66]:
rfm_customer = (
    rfm_customer
    .set_index("CustomerID")
    .loc[customer_ids_text]
    .reset_index()
)

assert (
    rfm_customer["CustomerID"].to_numpy() == customer_ids_text
).all()

print(rfm_customer.shape)
print(X_text_customer.shape)

(5879, 4)
(5879, 100)


In [67]:
print(rfm_customer.isna().sum())

print(
    "Duplicate customer IDs:",
    rfm_customer["CustomerID"].duplicated().sum()
)

print(
    "Customers in RFM:",
    rfm_customer["CustomerID"].nunique()
)

print(
    "Customers in text data:",
    len(customer_ids_text)
)

CustomerID    0
recency       0
frequency     0
monetary      0
dtype: int64
Duplicate customer IDs: 0
Customers in RFM: 5879
Customers in text data: 5879


In [68]:
rfm_customer[["recency", "frequency", "monetary"]].describe()

,recency,frequency,monetary
count,5879.000000,5879.000000,5.879000e+03
mean,201.297840,6.816976,3.567374e+03
std,209.337205,42.492965,4.458180e+04
min,1.000000,1.000000,2.950000e+00
25%,26.000000,1.000000,3.487750e+02
50%,96.000000,3.000000,8.989600e+02
75%,380.000000,7.000000,2.309050e+03
max,739.000000,3108.000000,3.229165e+06


### Behavioural features

In [69]:
invoice_timing = (
    purchases
    .groupby(["CustomerID", "InvoiceNo"], as_index=False)
    .agg(
        invoice_datetime=("InvoiceDate", "min"),
        invoice_time_5cat=(
            "invoice_time_5cat",
            lambda x: x.mode().iat[0]
        )
    )
)

invoice_timing['purchase_hour'] = (
    invoice_timing['invoice_datetime'].dt.hour
)

invoice_timing['weekday_num'] = (
    invoice_timing['invoice_datetime'].dt.dayofweek
)

invoice_timing['is_weekend'] = (
    invoice_timing['weekday_num'].isin([5, 6])
)

customer_time_profile = (
    invoice_timing
    .groupby('CustomerID')
    .agg(
        weekend_share=('is_weekend', 'mean'),
        avg_purchase_hour=('purchase_hour', 'mean'),
        preferred_hour=(
            'purchase_hour',
            lambda x: x.mode().iat[0]
        ),
        active_purchase_days=(
            'invoice_datetime',
            lambda x: x.dt.normalize().nunique()
        ),
        preferred_time_5cat=(
            'invoice_time_5cat',
            lambda x: x.mode().iat[0]        
        ),
    )
    .reset_index()
)

customer_time_profile.head()

,CustomerID,weekend_share,avg_purchase_hour,preferred_hour,active_purchase_days,preferred_time_5cat
0,12346.0,0.000,10.833333,13,8,midday
1,12347.0,0.125,12.500000,14,8,afternoon
2,12348.0,0.200,13.200000,10,5,midday
3,12349.0,0.000,9.750000,9,4,morning
4,12350.0,0.000,16.000000,16,1,afternoon


In [70]:
def average_days_between_orders(dates):
    ordered_dates = pd.Series(dates).sort_values().drop_duplicates()

    if len(ordered_dates) < 2:
        return np.nan

    return ordered_dates.diff().dropna().dt.total_seconds().div(
        86_400
    ).mean()

purchase_rhythm = (
    invoice_timing
    .groupby('CustomerID')['invoice_datetime']
    .agg(
        avg_days_between_orders=average_days_between_orders,
        order_date_count="nunique"
    )
    .reset_index()
)

purchase_rhythm.head()

,CustomerID,avg_days_between_orders,order_date_count
0,12346.0,36.369129,12
1,12347.0,57.437698,8
2,12348.0,90.731597,5
3,12349.0,190.284954,4
4,12350.0,NaN,1


### Aggregation of three Customer-Level Datasets

In [71]:
customer_profile = (
    rfm_customer
    .merge(customer_time_profile, on='CustomerID', how='left', validate='one_to_one')
    .merge(purchase_rhythm, on="CustomerID", how='left', validate='one_to_one')
    .merge(text_svd_df, on='CustomerID', how='left', validate='one_to_one')
)

customer_profile.head(5)

,CustomerID,recency,frequency,monetary,weekend_share,avg_purchase_hour,preferred_hour,active_purchase_days,preferred_time_5cat,avg_days_between_orders,...,text_svd_05,text_svd_06,text_svd_07,text_svd_08,text_svd_09,text_svd_10,text_svd_11,text_svd_12,text_svd_13,text_svd_14
0,12346.0,326,12,77556.46,0.000,10.833333,13,8,midday,36.369129,...,-0.054787,-0.021104,0.195492,0.084543,0.104115,-0.240231,0.245742,0.311689,0.199950,-0.054729
1,12347.0,2,8,5633.32,0.125,12.500000,14,8,afternoon,57.437698,...,-0.112825,0.064833,-0.152168,0.155788,-0.040855,-0.156677,0.061693,-0.027149,-0.060058,0.173658
2,12348.0,75,5,2019.40,0.200,13.200000,10,5,midday,90.731597,...,-0.161823,0.537691,-0.191536,0.081759,-0.091979,0.089767,-0.054764,0.083323,0.009273,0.111751
3,12349.0,19,4,4428.69,0.000,9.750000,9,4,morning,190.284954,...,-0.186275,-0.166294,0.324660,-0.083427,0.013956,-0.200510,0.100400,0.139637,-0.129949,-0.048644
4,12350.0,310,1,334.40,0.000,16.000000,16,1,afternoon,NaN,...,0.124714,-0.115467,0.113825,-0.004743,-0.085082,0.096691,0.024741,-0.027823,-0.030778,-0.002321


### Checks

In [72]:
customer_profile.isna().sum()

# Missing values avg_days_between_orders 

CustomerID                    0
recency                       0
frequency                     0
monetary                      0
weekend_share                 0
avg_purchase_hour             0
preferred_hour                0
active_purchase_days          0
preferred_time_5cat           0
avg_days_between_orders    1625
order_date_count              0
text_svd_00                   0
text_svd_01                   0
text_svd_02                   0
text_svd_03                   0
text_svd_04                   0
text_svd_05                   0
text_svd_06                   0
text_svd_07                   0
text_svd_08                   0
text_svd_09                   0
text_svd_10                   0
text_svd_11                   0
text_svd_12                   0
text_svd_13                   0
text_svd_14                   0
dtype: int64

In [73]:
customer_profile.describe()

,recency,frequency,monetary,weekend_share,avg_purchase_hour,preferred_hour,active_purchase_days,avg_days_between_orders,order_date_count,text_svd_00,...,text_svd_05,text_svd_06,text_svd_07,text_svd_08,text_svd_09,text_svd_10,text_svd_11,text_svd_12,text_svd_13,text_svd_14
count,5879.000000,5879.000000,5.879000e+03,5879.000000,5879.000000,5879.000000,5879.000000,4254.000000,5879.000000,5879.000000,...,5879.000000,5879.000000,5879.000000,5879.000000,5879.000000,5879.000000,5879.000000,5879.000000,5879.000000,5879.000000
mean,201.297840,6.816976,3.567374e+03,0.139721,12.641307,11.963769,5.724783,102.559841,6.746726,0.544875,...,-0.001035,-0.004770,-0.002714,0.005715,0.002202,-0.000686,0.003203,-0.000491,0.004270,0.001153
std,209.337205,42.492965,4.458180e+04,0.263779,1.743371,2.240601,12.255591,98.454007,39.586700,0.177617,...,0.163755,0.144615,0.140808,0.133913,0.129209,0.122722,0.120161,0.116351,0.114022,0.111976
min,1.000000,1.000000,2.950000e+00,0.000000,7.000000,6.000000,1.000000,0.000694,1.000000,0.000000,...,-0.418212,-0.379566,-0.418934,-0.449635,-0.469832,-0.450661,-0.406848,-0.407176,-0.342321,-0.479881
25%,26.000000,1.000000,3.487750e+02,0.000000,11.633971,10.000000,1.000000,38.777908,1.000000,0.432963,...,-0.115583,-0.108546,-0.098283,-0.078983,-0.084204,-0.073604,-0.075315,-0.076697,-0.072411,-0.061379
50%,96.000000,3.000000,8.989600e+02,0.000000,12.500000,12.000000,3.000000,72.566937,3.000000,0.557515,...,-0.013597,-0.018676,-0.017906,0.000000,-0.005902,0.000000,-0.003032,-0.011189,-0.003454,0.000000
75%,380.000000,7.000000,2.309050e+03,0.166667,13.666667,14.000000,6.000000,129.987847,7.000000,0.676360,...,0.099156,0.081397,0.078909,0.082463,0.087452,0.070408,0.073107,0.064355,0.073901,0.059495
max,739.000000,3108.000000,3.229165e+06,1.000000,20.000000,20.000000,549.000000,714.152083,2876.000000,0.975978,...,0.586948,0.649303,0.657676,0.520622,0.471069,0.529466,0.533331,0.550485,0.460213,0.540343


### Save Datasets on Customer-Level Information

In [74]:
## Purchases (main dataset)
customer_profile.to_csv('../01_data/02_processed_data/customer_profile.csv', index=False)
